In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")

In [3]:
df_ground_truth.head()

,question,document
0,"I just found this course, can I still join now...",74eb249bbf
1,Am I allowed to start the course after it has ...,74eb249bbf
2,"If I join late, can I still get a certificate ...",74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,"Is it okay to enroll now, even though I'm disc...",74eb249bbf


In [4]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
ground_truth[:5]

[{'question': 'I just found this course, can I still join now or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': "Is it okay to enroll now, even though I'm discovering the course late?",
  'document': '74eb249bbf'}]

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [11]:
boost={'question':3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [12]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [13]:
q = ground_truth[0]
q

{'question': 'I just found this course, can I still join now or is it too late?',
 'document': '74eb249bbf'}

In [14]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [15]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [16]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [17]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [19]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course, can I still join now or is it too late?


[1, 0, 0, 0, 0]

In [20]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where should I watch the live sessions for the course if the Zoom meeting isn’t shared with students?


[1, 0, 0, 0, 0]

In [21]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I keep track of the LLM Zoomcamp syllabus and deadlines in one place?


[1, 0, 0, 0, 0]

In [22]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [23]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [24]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [25]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [26]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [27]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [28]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [29]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [30]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [31]:
cnt / len(example)
# 0.933

0.9333333333333333

In [32]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [36]:
hit_rate(example)
# 0.933

0.9333333333333333

In [37]:
hit_rate(relevance_total)

0.8460176991150442

In [38]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

In [39]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [40]:
total_score / len(example)
# 0.822

0.8222222222222222

In [41]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [42]:
mrr(example)
# 0.822

0.8222222222222222

In [43]:
mrr(relevance_total)

0.7274336283185837

In [44]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [45]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8460176991150442, 'mrr': 0.7274336283185837}

In [46]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [47]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8920353982300885, 'mrr': 0.7934808259587016}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8920353982300885, 'mrr': 0.7871681415929199}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8460176991150442, 'mrr': 0.7274336283185837}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8194690265486726, 'mrr': 0.7013274336283182}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7893805309734513, 'mrr': 0.6648967551622417}


In [48]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [49]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [50]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.976991,0.879381
3,1.0,2.0,0.1,0.976991,0.879263
19,2.0,4.0,0.2,0.976991,0.879263
35,5.0,10.0,0.5,0.976991,0.879263
4,1.0,2.0,0.2,0.976991,0.879233
20,2.0,4.0,0.5,0.975221,0.879086
6,1.0,4.0,0.1,0.976991,0.879027
18,2.0,4.0,0.1,0.973451,0.878171
34,5.0,10.0,0.2,0.973451,0.877198
33,5.0,10.0,0.1,0.973451,0.876903


In [51]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 4.0,
        "section": 0.2,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )